# Processing data for publication

In [34]:
%matplotlib inline
import xarray as xr
import numpy as np
import cosima_cookbook as cc
from collections import OrderedDict
from dask.distributed import Client
import matplotlib.path as mpath
import os.path
import sys
sys.path.append(
    "/g/data/e14/cs6673/Ross_salinity/Python_scripts_published/") 
from Info_definitions import (
    path_database, path_output, path_plots, path_data_published,
    exptdict, shelf_mask_isobath, select_bottom_values, yearly_mean)
from gsw import p_from_z, SA_from_SP

# import cf_xarray
# from metpy.interpolate import cross_section
# import pyproj
# import xesmf
# from gsw import CT_from_pt, SA_from_SP, p_from_z, sigma2
# import matplotlib.pyplot as plt
# import cmocean.cm as cmo

In [2]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 28,Total memory: 251.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40409,Workers: 7
Dashboard: /proxy/8787/status,Total threads: 28
Started: Just now,Total memory: 251.19 GiB
Comm: tcp://127.0.0.1:35593,Total threads: 4
Dashboard: /proxy/41761/status,Memory: 35.88 GiB
Nanny: tcp://127.0.0.1:43225,


Exception during reset or similar
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.01/lib/python3.10/site-packages/sqlalchemy/pool/base.py", line 763, in _finalize_fairy
    fairy._reset(pool, transaction_was_reset)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.01/lib/python3.10/site-packages/sqlalchemy/pool/base.py", line 1038, in _reset
    pool._dialect.do_rollback(self)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.01/lib/python3.10/site-packages/sqlalchemy/engine/default.py", line 683, in do_rollback
    dbapi_connection.rollback()
sqlite3.ProgrammingError: SQLite objects created in a thread can only be used in that same thread. The object was created in thread id 22856958388032 and this is thread id 22855448332032.
Exception closing connection <sqlite3.Connection object at 0x14c94108d240>
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.01/lib/python3.10/sit

## Data for Figure 1

### Bathymetric features for maps

In [3]:
def mask_from_polygon(lon, lat, xt_ocean, yt_ocean):
    polygon = [(lon[0], lat[0])]
    for l in range(1, len(lon)):
        polygon += [(lon[l], lat[l])]
    poly_path = mpath.Path(polygon)

    x, y = xr.broadcast(xt_ocean, yt_ocean)
    coors = np.hstack((x.values.reshape(-1, 1), y.values.reshape(-1, 1)))

    mask = poly_path.contains_points(coors)
    mask = mask.reshape(xt_ocean.size, yt_ocean.size).transpose()
    mask = xr.DataArray(
        mask, dims=[yt_ocean.dims[0], xt_ocean.dims[0]],
        coords={xt_ocean.dims[0]: xt_ocean, yt_ocean.dims[0]: yt_ocean})
    return mask

In [6]:
# depth and mask for land
ht = cc.querying.getvar(
    '01deg_jra55v13_ryf9091_rerun_for_easterlies', 'ht',
    session=cc.database.create_session(), n=1).sel(
    yt_ocean=slice(-90, -60))
ht.name = 'depth'
ds = ht.to_dataset()
ds['land_mask'] = (ht*0).fillna(1)

# save dataset
comp = dict(zlib=True, complevel=5, shuffle=True)
enc = {var: comp for var in ds.data_vars}
ds.to_netcdf(path_data_published + 'Bathymetry_landmask.nc', encoding=enc)

mask_wind = [
    [360-167, 360-167, 360-140, 360-115, 360-115, 360-140, 360-167],
    [-80.5, -72, -69 , -69, -76, -77, -80.5]]
mask_DSW = mask_from_polygon(
    [185-360, 160-360, 164-360, 172-360, 185-360],
    [-78, -78, -73, -71.5, -78],
    land_mask.xt_ocean, land_mask.yt_ocean)
mask_DSW, shelf_mask = shelf_mask_isobath(mask_DSW, output_mask=True)
mask_DSW = mask_DSW.where(mask_DSW == True)
mask_DSW = mask_DSW.where(land_mask == 0)
mask_DSW = mask_DSW .where(mask_DSW == 1, 0)

### Wind forcing

In [8]:
def yearly_mean_Nov_Feb(data):
    # Compute the number of days in each month
    days_in_month = data['time'].dt.days_in_month
    
    # Select only the months Nov-Feb
    # Adjust year grouping for Nov-Feb to treat it as a single season
    nov_feb = data.sel(time=data['time'].dt.month.isin([11, 12, 1, 2]))
    
    # Assign season-year for Nov-Feb (shift the year for Jan-Feb back to Nov-Dec's year)
    season_year = nov_feb['time.year'].where(nov_feb['time.month'] > 2, nov_feb['time.year'] - 1)
    
    # Add the season-year coordinate for grouping
    nov_feb = nov_feb.assign_coords(season_year=season_year)
    
    # Calculate weighted values
    weights = days_in_month.sel(time=nov_feb['time'])
    weighted_data = nov_feb * weights
    
    # Aggregate weighted values over the Nov-Feb period for each season_year
    weighted_mean = (weighted_data.groupby('season_year').sum(dim='time') /
                     weights.groupby('season_year').sum(dim='time'))
    weighted_mean = weighted_mean.rename({'season_year': 'time'})
    weighted_mean.attrs = {'time average': 'seasonal (NOV-FEB) weighted mean'}
    
    # Resulting time series
    return weighted_mean

In [16]:
%%time
ekeys = ['ctrl', 'wind_50_down_zonal', 'era5', 'iaf']

for ekey in ekeys:
    e = exptdict[ekey]
    print(ekey)
    
    if ekey == 'era5':
        start_time = '1999-11-01'
        end_time = '2022-02-28'
        u = xr.open_mfdataset(e['u_file'], chunks='auto').sel(
            time=slice(start_time, end_time)).rename(
            {'latitude': 'lat', 'longitude': 'lon'}).u10
        # shift longitude from [-180, 180] to [0, 360]
        u['lon'] = (u['lon'] + 360) % 360
        u = u.sortby('lon')
        u = u[:, ::-1, :]  # reverse lat axis to range -90, 90
        e['u'] = yearly_mean_Nov_Feb(u)
    elif ekey == 'iaf':        
        start_time = '1981-11-01'
        end_time = '2022-02-28'
        u =  xr.open_mfdataset(
            '/g/data/qv56/replicas/input4MIPs/CMIP6Plus/OMIP/MRI/' +
            'MRI-JRA55-do-1-6-0/atmos/3hrPt/uas/gr/v20240531/' +
            'uas_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-6-0_gr*.nc',
            chunks='auto').uas
        u = u.sel(time=slice(start_time, end_time),
                  lat=slice(-82, -65), lon=slice(160, 290))
        e['u'] = yearly_mean_Nov_Feb(u)
    else:
        u =  xr.open_dataarray(e['u_file'], chunks='auto').rename(
            {'latitude': 'lat', 'longitude': 'lon'})
        v = xr.open_dataarray(e['v_file'], chunks='auto').rename(
            {'latitude': 'lat', 'longitude': 'lon'})
        u = u.sel(lat=slice(-82, -65), lon=slice(160, 290)).isel(
            time=u.time.dt.month.isin([11, 12, 1, 2]))
        v = v.sel(lat=slice(-82, -65), lon=slice(160, 290)).isel(
            time=v.time.dt.month.isin([11, 12, 1, 2]))
        e['u'] = u.mean('time').compute()
        e['v'] = v.mean('time').compute()
        e['spd'] = np.sqrt(e['u']**2 + e['v']**2).compute()

ctrl
wind_50_down_zonal
era5
iaf
CPU times: user 30.7 s, sys: 14.9 s, total: 45.7 s
Wall time: 2min 3s


In [33]:
# Simulated wind field
e = exptdict['ctrl']
u_ctrl = e['u']
u_ctrl.name = 'u_ctrl'
ds_wind = u_ctrl.to_dataset()
ds_wind['v_ctrl'] = e['v']
ds_wind['spd_ctrl'] = e['spd']

e = exptdict['wind_50_down_zonal']
ds_wind['u_wind_50_down_zonal'] = e['u']
ds_wind['v_wind_50_down_zonal'] = e['v']
ds_wind['spd_wind_50_down_zonal'] = e['spd']

# save dataset
comp = dict(zlib=True, complevel=5, shuffle=True)
enc = {var: comp for var in ds_wind.data_vars}
ds_wind.attrs = {
    'long_name': 'Zonal winds (Nov-Feb) averaged within red box in Figure 1c ' +
    ' in ERA5 and JRA55-do reanalysis data',
    'units': 'm s-1'}
ds_wind.to_netcdf(path_data_published + 'Wind_field_simulated.nc', encoding=enc)

In [34]:
# Time series of zonal wind in ERA5 and JRA55-do
buffer = -1.2
mask_wind_lon = (np.array([-167-buffer, -167-buffer, -140, -115+buffer,
                           -115+buffer, -140, -167-buffer]) + 360) % 360
mask_wind_lat = [-80-buffer, -72+buffer, -69+buffer , -69+buffer,
                 -76-buffer, -76-buffer, -80-buffer]
mask_wind_era = mask_from_polygon(
    mask_wind_lon, mask_wind_lat,
    exptdict['era5']['u'].lon, exptdict['era5']['u'].lat)
mask_wind_jra = mask_from_polygon(
    mask_wind_lon, mask_wind_lat,
    exptdict['ctrl']['u'].lon, exptdict['ctrl']['u'].lat)

era5_u = (exptdict['era5']['u'].where(mask_wind_era == 1).mean(['lon', 'lat']).sel(
    time=slice(2000, 2021)))
era5_u.name = 'era5_u'
ds_u = era5_u.to_dataset()
ds_u['jra55_u'] = (exptdict['iaf']['u'].where(mask_wind_jra == 1).mean(['lon', 'lat']).sel(
    time=slice(2000, 2021)))
ds_u.attrs = {
    'long_name': 'Timeseries of Zonal (u) and meridional (v) components and speed (spd) ' +
    'of the near-surface winds in the CONTROL (ctrl) and WIND- ' +
    '(wind_50_down_zonal) simulations',
    'units': 'm s-1'}
ds_u.to_netcdf(path_data_published + 'Timeseries_zonal_wind_era5_jra55.nc')

## Meltwater input

In [37]:
"""JRA forcing"""
runoff = xr.open_dataarray(
    '/g/data/ik11/inputs/JRA-55/RYF/v1-3/RYF.runoff_all.1990_1991.nc').sel(
    latitude=slice(-80, -60))
runoff = runoff.where(runoff != 0)
runoff = runoff.isel(time=0)  # select first time index as runoff is constant in Antarctica

# convert to common units
lat_length = np.ones((len(runoff.latitude), len(runoff.longitude)))*111/4*1e3 # make sure it's in m
lat_grid = np.pi/180*np.array([runoff.latitude.values]*len(runoff.longitude)).transpose()
lon_length = lat_length * np.cos(lat_grid)
area_grid = lat_length * lon_length
area_grid = xr.DataArray(area_grid, dims=['latitude', 'longitude'],
                         coords=[runoff.latitude, runoff.longitude])
runoff = runoff * area_grid # convert kg.m2/s to kg/s
rho_fw = 1000
# runoff_Sv = runoff * (1/rho_fw) * 1e-6  # Sv = kg/s * m^3/kg (1/rho_fw) * 10^-6
runoff_Gt = runoff*(60*60*24*365)*1e-12

# save dataset
runoff_Gt.name = 'runoff_Gt'
runoff_Gt.attrs = {
    'long_name': 'Meltwater input in CONTROL based on Deporter et al (2013)',
    'units': 'Gt yr-1'}
enc = {'runoff_Gt': {'zlib': True, 'complevel': 5, 'shuffle': True}}
runoff_Gt.to_netcdf(path_data_published + 'Meltwater_input_ctrl.nc',
                    encoding=enc)

In [42]:
""" Davison et al., 2023 """
path_obs = '/g/data/e14/cs6673/Ross_salinity/data_basal_melt/'
exptdict_glaciers = OrderedDict([
    ('Abbot', {'name': 'Abbot'}),
    ('Cosgrove', {'name': 'Cosgrove'}),
    ('Crosson', {'name': 'Crosson'}),
    ('Dotson', {'name': 'Dotson'}),
    ('Getz', {'name': 'Getz'}),
    ('Pine_Island', {'name': 'Pine_Island'}),
    ('Thwaites', {'name': 'Thwaites'}),
    ('all', {'name': 'all'})
])

ekeys = ['Abbot', 'Cosgrove', 'Crosson', 'Dotson', 'Getz',
         'Pine_Island', 'Thwaites']
for ekey in ekeys:
    e_glaciers = exptdict_glaciers[ekey]
    e_glaciers['df'] = pd.read_csv(path_obs + ekey + '-timeseries.csv')

rho_ice = 917  # Adusumilli et al., 2020: "assuming an ice density of 917 kg m–3"

for ekey in ekeys:
    e_glaciers = exptdict_glaciers[ekey]
    if ekey == ekeys[0]:
        bm_all = e_glaciers['df'].bm_monthly.copy()*e_glaciers['df'].area_km2*(1000**2)*rho_ice/1e12
        bm_err_all = e_glaciers['df'].bm_monthly_errors.copy()*e_glaciers['df'].area_km2*(1000**2)*rho_ice/1e12
        area_all = e_glaciers['df'].area_km2*(1000**2)
    bm_all += e_glaciers['df'].bm_monthly.copy()*e_glaciers['df'].area_km2*(1000**2)*rho_ice/1e12
    bm_err_all += e_glaciers['df'].bm_monthly_errors.copy()*e_glaciers['df'].area_km2*(1000**2)*rho_ice/1e12
    area_all += e_glaciers['df'].area_km2*(1000**2)
e_glaciers = exptdict_glaciers['all']
e_glaciers['melt'] = bm_all
e_glaciers['melt_err'] = bm_err_all
e_glaciers['area'] = area_all

# save Dataset
ds_Davison2023 = xr.Dataset(
    data_vars={'melt': ('time', e_glaciers['melt'].values),
               'melt_err': ('time', e_glaciers['melt_err'].values)},
    coords={'time': exptdict_glaciers['Getz']['df'].dates_decimal})
ds_Davison2023.attrs = {
    'long_name': 'Satellite-derived meltwater input and associated error from ' +
    'the Amundsen Sea ice shelves (namely the Abbot, Cosgrove, Crosson, Dotson, ' +
    'Getz, Pine Island, and Thwaites Glaciers) calculated based on Davison et al (2023)',
    'units': 'Gt yr-1'}
ds_Davison2023.to_netcdf(path_data_published + 'Meltwater_input_Davison2023.nc')

In [62]:
def to_decimal_year_np(timestamp):
    # Convert to datetime64 with day precision
    year = timestamp.astype('datetime64[Y]').astype(int) + 1970
    year_start = np.datetime64(f'{year}-01-01T00:00:00')
    year_end = np.datetime64(f'{year + 1}-01-01T00:00:00')

    # Compute elapsed and total seconds in the year
    elapsed_seconds = (timestamp - year_start) / np.timedelta64(1, 's')
    total_seconds = (year_end - year_start) / np.timedelta64(1, 's')

    # Compute decimal year
    return year + elapsed_seconds / total_seconds

In [66]:
""" Paolo et al 2023 """

ds_paolo = xr.open_dataset(
    path_obs + 'NSIDC-0792_19920317-20171216_V01.0.nc', chunks='auto')

# Transform melt data coordinates to lat-lon
inProj = Proj(init='epsg:3031') # the x-y coordinates of the dataset
outProj = Proj(init='epsg:4326') # regular lat-lon grid
x2d, y2d = np.meshgrid(ds_paolo.melt.x, ds_paolo.melt.y)
bas_lon2d,bas_lat2d = transform(inProj,outProj,x2d, y2d)
ds_paolo.coords['lat'] = (ds_paolo.melt_mean.dims, bas_lat2d)
ds_paolo.coords['lon'] = (ds_paolo.melt_mean.dims, bas_lon2d)

ds_paolo_AS = ds_paolo.where(
    (ds_paolo.lon > -150) & (ds_paolo.lon < -88) &
    (ds_paolo.lat > -77.7) & (ds_paolo.lat < -70.7))

dx = ds_paolo_AS.x.diff('x')
dy = ds_paolo_AS.y.diff('y')
area = np.abs(dx[0].values * dy[0].values)

# Melt rate in Paolo et al is in m/yr, transform to Gt/yr

# melt rate * area * ice density * 1e12
# m/yr * m2 * kg/m3 / 1e12 = Gt/yr

ds_Paolo2023 = (ds_paolo_AS.melt * area * rho_ice / 1e12).sum(['x', 'y']).compute()

ds_Paolo2023['time_decimal'] = xr.DataArray(
    np.zeros(len(ds_Paolo2023.time)), dims='time')
for l in range(len(ds_Paolo2023.time)):
    timestamp_np = ds_Paolo2023.time[l].values
    ds_Paolo2023.time_decimal[l] = to_decimal_year_np(timestamp_np)

In [67]:
ds_Paolo2023.attrs = {
    'long_name': 'Satellite-derived meltwater input at 88-150W calculated ' +
    'based on Paolo et al (2023)',
    'units': 'Gt yr-1'}
ds_Paolo2023.to_netcdf(path_data_published + 'Meltwater_input_Paolo2023.nc')

## Data for Figure 2

In [43]:
%%time
ekeys = ['ctrl', 'wind_50_down_zonal', 'mw_50_down']
start_time = '2150-01-01'
end_time = '2159-12-31'

area_t = cc.querying.getvar(
    exptdict['ctrl']['expt'], 'area_t',
    session=cc.database.create_session(), frequency='static',  n=1,
    chunks={'yt_ocean': '200MB', 'xt_ocean': '200MB'}).sel(
    yt_ocean=slice(-76, -71), xt_ocean=slice(-198, -184))

for ekey in ekeys:
    print(ekey)
    e = exptdict[ekey]
    if ekey == 'ctrl':
        session = cc.database.create_session()
    else:
        db = (path_database + e['expt'] + '.db')
        session = cc.database.create_session(db)

    """Salinity in southwestern Ross Sea"""
    salt = cc.querying.getvar(
        e['expt'], 'salt', session, frequency='1 monthly',
        start_time=start_time, end_time=end_time,
        chunks={'yt_ocean': '200MB', 'xt_ocean': '200MB'}).sel(
            yt_ocean=slice(-76, -71), xt_ocean=slice(-198, -184))
    # convert units to absolute salinity
    pressure = p_from_z(-salt.st_ocean, salt.yt_ocean)
    e['salt'] = SA_from_SP(
        salt, pressure, salt.xt_ocean, salt.yt_ocean).compute()

    for area in ['TNB', 'DT']:
        if area == 'TNB':
            # 74.75°S–75.50°S and 163.00°E–166.00°E, 870-900 dbar
            name = 'Terra Nova Bay'
            k = 44  # depth index 44 is appr. at depth of obs
            lon_range = [-197, -194]
            lat_range = [-75.5, -74.75]
        elif area == 'DT':
            # 72.00°S and 72.67°S and 171.50°E and 174.50°E,
            # bottom 20 dbar
            name = 'Drygalski Trough'
            k = 'bot'
            lon_range = [-188.5, -185.5]
            lat_range = [-72.67, -72]

        salt = e['salt'].sel(
            xt_ocean=slice(lon_range[0], lon_range[1]),
            yt_ocean=slice(lat_range[0], lat_range[1]))
        if k == 'bot':
            salt = select_bottom_values(salt)
        else:
            salt = salt.isel(st_ocean=k)
        area_weight = area_t.sel(
            xt_ocean=slice(lon_range[0], lon_range[1]),
            yt_ocean=slice(lat_range[0], lat_range[1]))
        area_weight = area_weight.where(np.isnan(salt[0, :]) == False)
        area_weight = area_weight/area_weight.sum(
            ['xt_ocean', 'yt_ocean'])
        
        salt = (salt * area_weight).sum(['xt_ocean', 'yt_ocean']).compute()

        if (ekey == 'ctrl') & (area == 'TNB'):
            salt.name = 'salt_TNB_ctrl'
            ds_salt = salt.to_dataset()
        else:
            ds_salt['salt_' + area + '_' + ekey] = salt



    """AABW thickness for rho>=27.86"""
    rho_AABW=27.86
    e['dzt'] = cc.querying.getvar(
        e['expt'], 'dzt', session, frequency='1 monthly',
        chunks={'yt_ocean': '100MB', 'xt_ocean': '100MB'}).sel(
        yt_ocean=slice(None, -60), xt_ocean=slice(-240, -60))

    e['rho0'] = cc.querying.getvar(
        e['expt'], 'pot_rho_0', session, frequency='1 monthly',
        start_time=start_time, end_time=end_time,
        chunks={'yt_ocean': '200MB', 'xt_ocean': '200MB'}).sel(
            yt_ocean=slice(None, -60), xt_ocean=slice(-240, -60)) - 1000

    thickAABW = e['dzt'].sel(
        xt_ocean=slice(-191.5, -189.5), yt_ocean=slice(-70.3, -69))
    thickAABW = yearly_mean(thickAABW.where(e['rho0'] >= rho_AABW).sum(
        'st_ocean').mean(['xt_ocean', 'yt_ocean'])).compute()
    if ekey == 'ctrl':
        thickAABW.name = 'AABW_thickness_ctrl'
        ds_thick = thickAABW.to_dataset()
    else:
        ds_thick['AABW_thickness_' + ekey] = thickAABW

# save data
ds_salt['time'] = np.linspace(0, len(salt.time)/12, len(salt.time))
ds_salt = ds_salt.drop_vars('st_ocean')
ds_salt.attrs = {
    'long_name': 'Absolute salinity in Terra Nova Bay (TNB) at 843 m ' +
    'and at the bottom of the Drygalsky Trough (DT) in the CONTROL (ctrl), ' +
    'WIND- (wind_50_down_zonal) and MELTWATER- (mw_50_down) simulations',
    'units': 'g kg-1'}
ds_salt.to_netcdf(path_data_published + 'Timeseries_salinity_Ross_Sea.nc')


ds_thick['time'] = np.arange(len(thickAABW.time)) + .5
ds_thick.attrs = {
    'long_name': 'Thickness of AABW with rho>= 27.86 kg m-3 off ' +
    'Cape Adare in the CONTROL (ctrl), WIND- (wind_50_down_zonal) ' +
    'and MELTWATER- (mw_50_down) simulations',
    'units': 'm'}
ds_thick.to_netcdf(path_data_published + 'Timeseries_AABW_thickness_Ross_Sea.nc')

ctrl
wind_50_down_zonal
mw_50_down
CPU times: user 49.7 s, sys: 8.34 s, total: 58 s
Wall time: 2min 9s


In [59]:
"""Surface Water Mass Transformation (SWMT) in southwestern Ross Sea"""
ekeys = ['ctrl', 'wind_50_down_zonal', 'mw_50_down']
for ekey in ekeys:
    e = exptdict[ekey]
    ds_SWMT = xr.open_mfdataset(
        path_output + 'SWMT_in_AABW_formation_region_' + e['expt']  +
        '_1m_????.nc')
    ds_SWMT = ds_SWMT.sel(area='Ross').sel(
        time=slice(start_time[:4], end_time[:4]))
    e['swmt'] = yearly_mean(
        ds_SWMT.binned_salt_transformation_in_AABW_region +
        ds_SWMT.binned_heat_transformation_in_AABW_region).compute()

    # Is the SWMT mostly influenced by surface fluxes
    # or the background density field?
    if ekey != 'ctrl':
        ds_SWMT = xr.open_mfdataset(
            path_output + 'SWMT_in_AABW_formation_region_' + e['expt']  +
            '_fluxes_' + exptdict['ctrl']['expt'] + '_rho0_1m_????.nc')
        ds_SWMT = ds_SWMT.sel(area='Ross').sel(
            time=slice(start_time[:4], end_time[:4]))
        e['swmt_ctrl_rho'] = yearly_mean(
            ds_SWMT.binned_salt_transformation_in_AABW_region +
            ds_SWMT.binned_heat_transformation_in_AABW_region).compute()

        ds_SWMT = xr.open_mfdataset(
            path_output + 'SWMT_in_AABW_formation_region_' +
            exptdict['ctrl']['expt'] + '_fluxes_' + e['expt'] + 
            '_rho0_1m_????.nc')
        ds_SWMT = ds_SWMT.sel(area='Ross').sel(
            time=slice(start_time[:4], end_time[:4]))
        e['swmt_ctrl_fluxes'] = yearly_mean(
            ds_SWMT.binned_salt_transformation_in_AABW_region +
            ds_SWMT.binned_heat_transformation_in_AABW_region).compute()


# Mean SWMT in model years 6-10
time_slice=slice('2155', '2159')
ekeys = ekeys[1:]
    
e = exptdict['ctrl']
# maximum SWMT of time mean 
swmt_max = e['swmt'].mean('time')[e['swmt'].mean('time').argmax(
    'isopycnal_bins').values]
# isopycnal bin corresponding to SWMT 75% and 25% below its maximum value
swmt_sig_bin_75 = e['swmt'].isopycnal_bins.sel(
    isopycnal_bins=slice(swmt_max.isopycnal_bins, None)).where(
    e['swmt'].mean('time') <= swmt_max*.75).min('isopycnal_bins')
swmt_sig_bin_25 = e['swmt'].isopycnal_bins.sel(
    isopycnal_bins=slice(swmt_max.isopycnal_bins, None)).where(
    e['swmt'].mean('time') <= swmt_max*.25).min('isopycnal_bins')

for i, ekey in enumerate(ekeys):
    e = exptdict[ekey]
    var_all = ['swmt', 'swmt', 'swmt_ctrl_rho', 'swmt_ctrl_fluxes']

    for p, var in enumerate(var_all):
        if p == 0:
            swmt = exptdict['ctrl'][var].sel(time=time_slice)
        else:
            swmt = e[var].sel(time=time_slice)
        swmt = swmt.mean('time')
        swmt_DSW = swmt.sel(isopycnal_bins=slice(
            swmt_sig_bin_75, swmt_sig_bin_25)).mean('isopycnal_bins')
        # apply rolling mean as sigma bin spacing is very high now
        swmt = swmt.rolling(isopycnal_bins=4, center=True).mean()

        if (i == 0) & (p == 0):
            swmt.name = 'swmt_ctrl'
            ds_swmt = swmt.to_dataset()
        else:
            if len(var) == 4:
                ds_swmt['swmt_' + ekey] = swmt
            else:
                ds_swmt['swmt_' + ekey + '_with' + var[4:]] = swmt

# save data
ds_swmt.attrs = {
    'long_name': 'Surface_water_mass_transformation (swmt) in the ' +
    'southwestern Ross Sea averaged over model year 6-10 in the ' +
    'CONTROL (ctrl), WIND- (wind_50_down_zonal) and MELTWATER- (mw_50_down) ' +
    'simulations. Additionally, the swmt is recalculated using either the surface ' +
    'density field of the CONTROL experiment and surface fluxes from the perturbation ' +
    'experiments (...with_ctrl_rho) or the opposite (...with_ctrl_fluxes; i.e., ' +
    'surface density from the perturbation experiments and surface fluxes from the CONTROL)',
    'units': 'Sv'}
ds_swmt.to_netcdf(path_data_published + 'Surface_water_mass_transformation_Ross_Sea.nc')